# 24 — Descriptive Statistics

Quantitative and categorical summaries of the analysis corpus (610 hotels /
174,371 reviews), reported **separately per source** because Agoda and Google Maps
expose different review metadata:

- **Google Maps** carries per-aspect sub-ratings (room / service / location, 1–5),
  reviewer activity (review & photo counts), trip type and travel group.
- **Agoda** carries a single overall score (0–10), stay length, and traveller type.

Each block gives a **quantitative** table (N / Mean / SD / Min / Median / Max) and a
**categorical** table (group / n / %). No silver-label fields are used.

### Source of variables

**Hotel-level attributes are taken uniformly from the Agoda partner catalogue**
(`hotel.csv`, an Agoda affiliate property feed): `star_rating`, `numberrooms`,
`accommodation_type`, `chain`/`brand`, city, geolocation, and the derived
`distance2coastline`. Google Maps contributed **only review-level content** (text,
rating, aspect tags, reviewer info) — no hotel metadata. Google has no official star
class, so the Agoda star rating is applied to its reviews too.

Consequently, hotel class is measured **identically** across both platforms: any
difference between the Google (G2) and Agoda (A2) star distributions reflects *which
hotels each platform reviews* (Google covers 417 hotels, Agoda 314, overlap 121),
**not** a difference in how stars were defined.

> Cleaning applied: Google `tag_room/service/location` kept only when in 1–5
> (scraping junk like 1203 dropped); Vietnamese category labels mapped to English
> with trailing `…` truncations collapsed by prefix. `star_rating = 0` means
> *unrated* in the Agoda catalogue (9 hotels), not a literal zero.

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

DB_PATH  = Path("../data/hotel_reviews.db")
FILT_CSV = Path("../data-management/data/hotel_filtered.csv")
filt = pd.read_csv(FILT_CSV, usecols=["hotel_id"])

con = duckdb.connect(str(DB_PATH), read_only=True)
con.register("filt", filt)


def quant_table(specs: list) -> pd.DataFrame:
    """Quantitative summary table. specs = [(label, series), ...]."""
    rows = []
    for label, s in specs:
        s = pd.to_numeric(s, errors="coerce")
        rows.append([label, int(s.notna().sum()), round(s.mean(), 2), round(s.std(), 2),
                     round(s.min(), 2), round(s.median(), 2), round(s.max(), 2)])
    return pd.DataFrame(rows, columns=["Variable", "N", "Mean", "SD", "Min", "Median", "Max"])


def cat_table(specs: list) -> pd.DataFrame:
    """Categorical summary table (stacked). specs = [(var_label, series), ...]."""
    out = []
    for label, s in specs:
        vc = s.value_counts(dropna=True)
        tot = vc.sum()
        for grp, n in vc.items():
            out.append([label, grp, int(n), round(n / tot * 100, 1)])
    return pd.DataFrame(out, columns=["Variable", "Group", "n", "%"])


def clean_map(series: pd.Series, mapping: list, default="Other") -> pd.Series:
    """Map Vietnamese category labels to English by prefix (handles '…' truncation)."""
    def m(v):
        if pd.isna(v):
            return np.nan
        s = str(v).replace("…", "").strip()
        for prefix, eng in mapping:
            if s.startswith(prefix):
                return eng
        return default
    return series.map(m)


def star_label(x):
    """Merge hotel star class to whole-star buckets: half-stars rounded up, 0 -> 'Unrated'."""
    if pd.isna(x):
        return np.nan
    if x == 0:
        return "Unrated"
    return f"{int(np.ceil(x))}-star"


BAND_BINS, BAND_LABELS = [0, 0.1, 1.0, np.inf], ["A <0.1km", "B 0.1-1km", "C >=1km"]
print("Connected.")

## A. Hotel level (supply side, shared across both sources) — 610 hotels

In [ ]:
hotels = con.execute("""
    SELECT h.*
    FROM HOTEL h
    WHERE h.hotel_id IN (SELECT hotel_id FROM filt)
      AND h.hotel_id IN (SELECT DISTINCT hotel_id FROM REVIEW_DATA)
""").df()
hotels["coast_band"] = pd.cut(hotels["distance2coastline"], BAND_BINS, right=False, labels=BAND_LABELS)
hotels["acc_type"]   = hotels["accommodation_type"].where(hotels["accommodation_type"] == "Hotel", "Other (mixed)")
print(f"Hotels: {len(hotels):,}\n")

print("TABLE H1 — Hotel level, quantitative")
display(quant_table([
    ("Hotel class (stars)", hotels.star_rating),
    ("Number of rooms",     hotels.numberrooms),
    ("Year opened",         hotels.yearopened),
    ("Distance to coast (km)", hotels.distance2coastline),
]))

print("\nTABLE H2 — Hotel level, categorical")
display(cat_table([
    ("Accommodation type", hotels.acc_type),
    ("City",               hotels.city),
    ("Coastal band",       hotels.coast_band),
]))

## B. Google Maps reviews (n = 112,242)

Richer aspect metadata: overall rating plus room / service / location sub-ratings
(each 1–5, ~30 % of reviews), reviewer activity, trip type and travel group.

In [ ]:
g = con.execute("""
    SELECT g.rating,
           CASE WHEN TRY_CAST(g.tag_room AS DOUBLE)     BETWEEN 1 AND 5 THEN TRY_CAST(g.tag_room AS DOUBLE)     END AS room_rating,
           CASE WHEN TRY_CAST(g.tag_service AS DOUBLE)  BETWEEN 1 AND 5 THEN TRY_CAST(g.tag_service AS DOUBLE)  END AS service_rating,
           CASE WHEN TRY_CAST(g.tag_location AS DOUBLE) BETWEEN 1 AND 5 THEN TRY_CAST(g.tag_location AS DOUBLE) END AS location_rating,
           g.reviewer_reviews, g.reviewer_photos, g.image_count,
           g.tag_trip_type, g.tag_travel_group, g.is_local_guide, g.language,
           LENGTH(TRIM(g.review_text)) - LENGTH(REPLACE(TRIM(g.review_text), ' ', '')) + 1 AS review_words,
           h.star_rating, h.distance2coastline
    FROM GOOGLEMAPS_REVIEW g JOIN HOTEL h USING (hotel_id)
    WHERE g.hotel_id IN (SELECT hotel_id FROM filt)
""").df()
print(f"Google Maps reviews: {len(g):,}\n")

print("TABLE G1 — Google Maps, quantitative")
display(quant_table([
    ("Overall rating (1–5)",  g.rating),
    ("Room rating (1–5)",     g.room_rating),
    ("Service rating (1–5)",  g.service_rating),
    ("Location rating (1–5)", g.location_rating),
    ("Reviewer's # reviews",       g.reviewer_reviews),
    ("Reviewer's # photos",        g.reviewer_photos),
    ("Images attached to review",  g.image_count),
    ("Review length (words)",      g.review_words),
    ("Distance to coast (km)",     g.distance2coastline),
]))

g["trip"]  = clean_map(g.tag_trip_type,    [("Chuyến nghỉ", "Leisure"), ("Chuyến công", "Business")])
g["grp"]   = clean_map(g.tag_travel_group, [("Gia đình", "Family"), ("Cặp đôi", "Couple"), ("Bạn bè", "Friends"), ("Khách lẻ", "Solo")])
g["lg"]    = g.is_local_guide.map({True: "Local Guide", False: "Regular user"})
g["star"]  = g.star_rating.map(star_label)
g["band"]  = pd.cut(g.distance2coastline, BAND_BINS, right=False, labels=BAND_LABELS)

print("\nTABLE G2 — Google Maps, categorical")
display(cat_table([
    ("Hotel class",   g.star),
    ("Trip type",     g.trip),
    ("Travel group",  g.grp),
    ("Local Guide",   g.lg),
    ("Language",      g.language),
    ("Coastal band",  g.band),
]))

## C. Agoda reviews (n = 62,129)

A single overall score (0\u201310), stay length, and the traveller `group_type`;
no per-aspect sub-scores are provided by Agoda.

In [ ]:
a = con.execute("""
    SELECT a.score, a.stay_nights, a.group_type, a.reviewer_continent, a.language,
           LENGTH(TRIM(a.review_text)) - LENGTH(REPLACE(TRIM(a.review_text), ' ', '')) + 1 AS review_words,
           h.star_rating, h.distance2coastline
    FROM AGODA_REVIEW a JOIN HOTEL h USING (hotel_id)
    WHERE a.hotel_id IN (SELECT hotel_id FROM filt)
""").df()
print(f"Agoda reviews: {len(a):,}\n")

print("TABLE A1 — Agoda, quantitative")
display(quant_table([
    ("Overall score (0–10)",  a.score),
    ("Stay length (nights)",       a.stay_nights),
    ("Review length (words)",      a.review_words),
    ("Distance to coast (km)",     a.distance2coastline),
]))

a["guest"] = clean_map(a.group_type, [("Cặp đôi", "Couple"), ("Du lịch một mình", "Solo"), ("Gia đình", "Family"), ("Nhóm", "Group"), ("Đi công tác", "Business"), ("Công tác", "Business")])
a["star"]  = a.star_rating.map(star_label)
a["band"]  = pd.cut(a.distance2coastline, BAND_BINS, right=False, labels=BAND_LABELS)

print("\nTABLE A2 — Agoda, categorical")
display(cat_table([
    ("Hotel class",        a.star),
    ("Guest type",         a.guest),
    ("Reviewer continent", a.reviewer_continent),
    ("Language",           a.language),
    ("Coastal band",       a.band),
]))

In [ ]:
con.close()
print("Done.")